# CUDA 编译运行模板 (Colab / T4)

**用法**:
1. 菜单 `代码执行程序` → `更改运行时类型` → 硬件加速器选 **T4 GPU**。
2. 从上到下依次运行各 cell。
3. 第 2 步会 clone/pull `cuda-learning` 仓库拿到最新的 `common.h`;第 3 步的 `%%writefile` cell 里粘贴你的 kernel 代码。

> 本机(Mac Mini)不能跑 nvcc,所有编译运行都在这里做。编译目标 `-arch=sm_75`(T4, Turing)。

## 1. 确认 GPU 环境

In [ ]:
!nvidia-smi
!nvcc --version

## 2. 拉取仓库获取 common.h

从 GitHub 仓库 clone/pull,始终拿到与本地仓库同步的最新版本,不需要手动上传或粘贴。

In [ ]:
import os

repo_dir = "cuda-learning"
if os.path.isdir(repo_dir):
    !git -C {repo_dir} pull
else:
    !git clone https://github.com/saintlion/cuda-learning.git {repo_dir}

!cp {repo_dir}/week1/common.h .

## 3. 写 kernel 代码

把你的程序写进 `main.cu`。记得每个 kernel 程序要有:**错误检查宏 + 边界检查 + CPU 端验证**。

In [ ]:
%%writefile main.cu
#include "common.h"
#include <cstdio>

// TODO: 在这里写你的 kernel 和 main()

int main() {
    printf("hello from host\n");
    return 0;
}

## 4. 编译

`-arch=sm_75` 对应 T4 (Turing)。如果要在 GTX 1050 Ti (sm_61) 上跑,注意别用 Turing+ 独有特性。
`-lineinfo` 让 profiler 能把指令映射回源码行;`-O2` 常规优化。

In [ ]:
!nvcc -arch=sm_75 -O2 -lineinfo main.cu -o main 2>&1

## 5. 运行

In [ ]:
!./main

## 6.(可选)compute-sanitizer 查内存越界 / 竞态

类比嵌入式里的 Valgrind / 硬件 MPU 报错。kernel 越界、未初始化读、竞态在这里能抓到。

In [ ]:
!compute-sanitizer --tool memcheck ./main